# Pothole & Road Surface Damage — Kaggle Pipeline

Mirrors `scripts/` end to end on Kaggle's GPU runtime, where the RDD2022 data and CUDA actually live.
Local (Apple Silicon / MPS) is for iterating on script logic only — real training runs happen here.

Importing just this notebook via Kaggle's GitHub integration does NOT bring along `scripts/` —
Kaggle only pulls the single `.ipynb`. The next cell clones the repo instead. Attach the RDD2022
dataset as an Input separately.

In [ ]:
import os

REPO_DIR = "/kaggle/working/pothole_cv"
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 https://github.com/kabirarshafidz/pothole_cv.git {REPO_DIR}

%pip install -q -r {REPO_DIR}/requirements-kaggle.txt

In [ ]:
import sys

sys.path.append(f"{REPO_DIR}/scripts")

import torch

from utils import CLASS_NAMES, COUNTRIES, DATA_ROOT, OUTPUTS_ROOT, RUNS_ROOT, get_device

print("device:", get_device())
print("cuda available:", torch.cuda.is_available())
print("DATA_ROOT:", DATA_ROOT)

## 1. Data prep

Convert Pascal VOC XML → YOLO labels, then split 70/15/15 (seed 42). Run once per country;
re-running `split_dataset` after training has started causes data leakage.

In [ ]:
from convert_annotations import convert_one
from split_dataset import split_country

for xml_path in sorted((DATA_ROOT / "Japan" / "annotations" / "xmls").glob("*.xml")):
    label_dir = DATA_ROOT / "Japan" / "labels"
    label_dir.mkdir(parents=True, exist_ok=True)
    convert_one(xml_path, label_dir / f"{xml_path.stem}.txt")

split_country("Japan")

In [ ]:
import yaml

dataset_yaml = {
    "path": str(DATA_ROOT / "Japan"),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
}
dataset_yaml_path = "/kaggle/working/dataset_japan.yaml"
with open(dataset_yaml_path, "w") as f:
    yaml.safe_dump(dataset_yaml, f)

print(dataset_yaml)

## 2. Train (YOLOv8s, Japan baseline)

Expected mAP@50: 0.60–0.67. Above 0.80 almost certainly means train/test overlap — recheck the split.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
model.train(
    data=dataset_yaml_path,
    imgsz=640,
    batch=16,
    epochs=50,
    patience=10,
    device=get_device(),
    project=str(RUNS_ROOT),
    name="japan_baseline",
)

best_weights = RUNS_ROOT / "japan_baseline" / "weights" / "best.pt"

## 3. Cross-country generalization

Evaluate the Japan-trained model on all 6 countries. Expect a 15–25 mAP point drop
Japan → India, with D20 (alligator crack) transferring worst — that's the core finding, not a bug.

In [ ]:
import csv

eval_model = YOLO(str(best_weights))
OUTPUTS_ROOT.mkdir(parents=True, exist_ok=True)
results_path = OUTPUTS_ROOT / "generalization_results.csv"

with open(results_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["country", "map50", "map50_95", "precision", "recall"])
    for country in COUNTRIES:
        metrics = eval_model.val(data=str(DATA_ROOT / country / "dataset.yaml"), device=get_device())
        writer.writerow([country, metrics.box.map50, metrics.box.map, metrics.box.mp, metrics.box.mr])

print(f"Wrote {results_path}")

## 4. Severity scoring (area vs depth)

Compares the area-based baseline against Depth Anything-assisted severity on a sample test image.

In [ ]:
from severity import area_severity, depth_severity
from PIL import Image
from transformers import pipeline
import numpy as np

sample_image_path = next((DATA_ROOT / "Japan" / "images" / "test").glob("*.jpg"))
sample_image = Image.open(sample_image_path).convert("RGB")
detections = eval_model.predict(sample_image, device=get_device())[0]

depth_pipe = pipeline("depth-estimation", model="LiheYoung/depth-anything-small-hf", device=get_device())
depth_map = np.array(depth_pipe(sample_image)["depth"])

for box in detections.boxes:
    xyxy = box.xyxy[0].tolist()
    a = area_severity(xyxy, sample_image.size[::-1])
    d = depth_severity(xyxy, depth_map)
    print(f"class={int(box.cls[0])} area_severity={a:.4f} depth_severity={d:.4f}")

## 5. Demo visualization

Annotated image with severity-colored boxes (green=low, orange=medium, red=high).

In [ ]:
import cv2
import matplotlib.pyplot as plt

from visualize import SEVERITY_COLORS, bucket

image_bgr = cv2.imread(str(sample_image_path))
for box in detections.boxes:
    x1, y1, x2, y2 = (int(v) for v in box.xyxy[0].tolist())
    score = area_severity((x1, y1, x2, y2), image_bgr.shape[:2])
    color = SEVERITY_COLORS[bucket(score)]
    cv2.rectangle(image_bgr, (x1, y1), (x2, y2), color, 2)

plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()